In [1]:
# USE A100
import os
		
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds = Dataset.load_from_disk("../data/alpaca_data_zh/")
ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 26858
})

In [4]:
ds = ds.train_test_split(test_size=0.2)
ds['train'][:3]

{'output': ['这些食材可以用来做许多不同的食物，但它们常见于糕点或甜点的制作，比如蛋糕或薄饼。',
  '目标受众：工作者或正在工作的人。',
  '在英语中，与 "引出" 意思相似的单词有 "introduce", "start", "begin", "commence", "initiate", "precede", "kick off" 等。它们都表示开始或引导某件事。'],
 'input': ['输入：牛奶、糖、鸡蛋、香草、面粉。', '输入：暂停工作，享受一碗冰淇淋。', ''],
 'instruction': ['识别出给定的食材是什么食物。 ', '确定陈述句的目标受众。 ', '找到文章中与“引出”意思相似的单词。']}

In [5]:
tokenizer = AutoTokenizer.from_pretrained("/tmp/code/chatglm3-6b", trust_remote_code=True)
tokenizer

ChatGLMTokenizer(name_or_path='/tmp/code/chatglm3-6b', vocab_size=64798, model_max_length=1000000000000000019884624838656, is_fast=False, padding_side='left', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<unk>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	
}
)

参考修改：https://blog.csdn.net/YiZhiYeShenJun/article/details/148948646

In [6]:
tokenizer.eos_token_id, tokenizer(tokenizer.eos_token)

(2,
 {'input_ids': [64790, 64792, 2893, 30917, 30994], 'attention_mask': [1, 1, 1, 1, 1], 'position_ids': [0, 1, 2, 3, 4]})

In [7]:
def process_func(example):
    MAX_LENGTH = 256
    input_ids, attention_mask, labels = [], [], []
    instruction = "\n".join([example["instruction"], example["input"]]).strip()     # query
    instruction = tokenizer.build_chat_input(instruction, history=[], role="user")  # [gMASK]sop<|user|> \n query<|assistant|>
    response = tokenizer("\n" + example["output"], add_special_tokens=False)        # \n response, 缺少eos token
    input_ids = instruction["input_ids"][0].numpy().tolist() + response["input_ids"] + [tokenizer.eos_token_id]
    attention_mask = instruction["attention_mask"][0].numpy().tolist() + response["attention_mask"] + [1]
    labels = [-100] * len(instruction["input_ids"][0].numpy().tolist()) + response["input_ids"] + [tokenizer.eos_token_id]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [8]:
tokenized_ds = ds.map(process_func, remove_columns=ds['train'].column_names)
tokenized_ds

Map: 100%|██████████| 5372/5372 [00:02<00:00, 1821.84 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 21486
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 5372
    })
})

In [9]:
tokenizer.decode(tokenized_ds['train'][1]["input_ids"])

'[gMASK]sop<|user|> \n 确定陈述句的目标受众。 \n输入：暂停工作，享受一碗冰淇淋。<|assistant|> \n目标受众：工作者或正在工作的人。'

In [10]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds['train'][1]["labels"])))

'\n目标受众：工作者或正在工作的人。'

In [11]:
import torch

"""
新版本中需要将modeling_chatglm源码中的613行部分进行调整，代码如下：

```
if not kv_caches:
    kv_caches = [None for _ in range(self.num_layers)]
else:
    kv_caches = kv_caches[1]
```

如果不进行调整，后续chat阶段会报错
"""
# 多卡情况，可以去掉device_map="auto"，否则会将模型拆开
model = AutoModelForCausalLM.from_pretrained("/tmp/code/chatglm3-6b", trust_remote_code=True, torch_dtype=torch.bfloat16)

Loading checkpoint shards: 100%|██████████| 7/7 [00:12<00:00,  1.84s/it]


In [14]:
for name, param in model.named_parameters():
    print(name)

transformer.embedding.word_embeddings.weight
transformer.encoder.layers.0.input_layernorm.weight
transformer.encoder.layers.0.self_attention.query_key_value.weight
transformer.encoder.layers.0.self_attention.query_key_value.bias
transformer.encoder.layers.0.self_attention.dense.weight
transformer.encoder.layers.0.post_attention_layernorm.weight
transformer.encoder.layers.0.mlp.dense_h_to_4h.weight
transformer.encoder.layers.0.mlp.dense_4h_to_h.weight
transformer.encoder.layers.1.input_layernorm.weight
transformer.encoder.layers.1.self_attention.query_key_value.weight
transformer.encoder.layers.1.self_attention.query_key_value.bias
transformer.encoder.layers.1.self_attention.dense.weight
transformer.encoder.layers.1.post_attention_layernorm.weight
transformer.encoder.layers.1.mlp.dense_h_to_4h.weight
transformer.encoder.layers.1.mlp.dense_4h_to_h.weight
transformer.encoder.layers.2.input_layernorm.weight
transformer.encoder.layers.2.self_attention.query_key_value.weight
transformer.enco

In [15]:
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

config = LoraConfig(target_modules=["query_key_value"], modules_to_save=["post_attention_layernorm"])
config

LoraConfig(task_type=None, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'query_key_value'}, exclude_modules=None, lora_alpha=8, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=['post_attention_layernorm'], init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False)

In [16]:
model = get_peft_model(model, config)

/usr/local/lib/python3.11/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)


In [17]:
for name, parameter in model.named_parameters():
    print(name)

base_model.model.transformer.embedding.word_embeddings.weight
base_model.model.transformer.encoder.layers.0.input_layernorm.weight
base_model.model.transformer.encoder.layers.0.self_attention.query_key_value.base_layer.weight
base_model.model.transformer.encoder.layers.0.self_attention.query_key_value.base_layer.bias
base_model.model.transformer.encoder.layers.0.self_attention.query_key_value.lora_A.default.weight
base_model.model.transformer.encoder.layers.0.self_attention.query_key_value.lora_B.default.weight
base_model.model.transformer.encoder.layers.0.self_attention.dense.weight
base_model.model.transformer.encoder.layers.0.post_attention_layernorm.original_module.weight
base_model.model.transformer.encoder.layers.0.post_attention_layernorm.modules_to_save.default.weight
base_model.model.transformer.encoder.layers.0.mlp.dense_h_to_4h.weight
base_model.model.transformer.encoder.layers.0.mlp.dense_4h_to_h.weight
base_model.model.transformer.encoder.layers.1.input_layernorm.weight
ba

In [19]:
model.print_trainable_parameters()

trainable params: 2,064,384 || all params: 6,245,648,384 || trainable%: 0.0331


In [20]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    logging_steps=50,
    eval_strategy='steps',
    num_train_epochs=1,
    learning_rate=1e-4,
    remove_unused_columns=False,
    save_strategy="epoch"
)

In [21]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds['train'].select(range(5000)),
    eval_dataset=tokenized_ds['test'].select(range(1000)),
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

Detected kernel version 4.15.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


[2025-12-16 09:52:38,409] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/local/lib/python3.11/site-packages/deepspeed/ops/op_builder/builder.py:18: DeprecationWarning: The distutils.sysconfig module is deprecated, use sysconfig instead
  import distutils.sysconfig
df: /root/.triton/autotune: 没有那个文件或目录


[2025-12-16 09:52:40,422] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


In [22]:
trainer.train()

Step,Training Loss,Validation Loss
50,2.392700,2.137781
100,2.030900,1.973734
150,1.944100,1.889906
200,1.928800,1.871125
250,1.872300,1.862344
300,1.905600,1.858750


TrainOutput(global_step=313, training_loss=1.9995101587460065, metrics={'train_runtime': 371.8364, 'train_samples_per_second': 13.447, 'train_steps_per_second': 0.842, 'total_flos': 2.226785959305216e+16, 'train_loss': 1.9995101587460065, 'epoch': 1.0})

参考：https://blog.csdn.net/qq_43749831/article/details/146022858

In [23]:
model.eval()
response, history = model.chat(tokenizer, "你好", history=[])
print(response)

你好！有什么我能为你效劳的吗？


In [24]:
response, history = model.chat(tokenizer, "晚上睡不着应该怎么办", history=history)
print(response)

失眠是一种常见的问题，可以尝试以下方法来改善睡眠：

1. 制定一个固定的睡眠时间表，尽量每天在相同的时间入睡和起床。
2. 建立一个放松的睡前习惯，如冥想、深呼吸、温暖的淋浴等。
3. 避免在睡前过度使用电子设备，如手机、电视等，这些设备会发出蓝光，影响睡眠质量。
4. 避免摄入过多的咖啡因和酒精，这些都可能导致失眠。
5. 尝试进行一些轻柔的伸展运动，如瑜伽、普拉提等，有助于放松身体。

如果以上方法不能解决你的失眠问题，建议咨询医生或专业的睡眠医学专家，他们可以提供更具体的建议和治疗方案。


In [25]:
print(model.chat(tokenizer, "数学考试怎么考高分？", history=history)[0])

数学考试要想取得高分，需要掌握一些基本的策略和技巧，具体可以参考下述建议：
1. 认真复习考试大纲所涉及的知识点，并确保理解每个概念、公式和定理，同时注意掌握解题技巧和策略。
2. 做练习题和模拟试题，并分析自己的错误和不足之处，以便在考试中避免犯同样的错误。
3. 熟悉考试的题型和考试要求，以便在考试中能够迅速、准确地理解题目，并迅速找到解题思路。
4. 在考试中注意审题，确保理解题目要求，并注意细节，如题目中的限制条件和 assumptions。
5. 在考试中避免过度紧张和焦虑，保持冷静和自信，并尝试控制自己的呼吸和情绪。
6. 尝试在考试前放松自己，如做一些轻松的活动、听音乐或做一些冥想等，以便在考试中保持良好的精神状态。

总之，要想在数学考试中取得高分，需要认真复习、练习、控制情绪，并注意细节和审题。
